## Perceptron-based Classification from Scratch

### What is a Perceptron?

A **Perceptron** is the simplest form of an artificial neural network, acting as a binary linear classifier. It's inspired by the biological neuron and is a fundamental building block for more complex neural networks.

At its core, a perceptron takes multiple binary (or real-valued) inputs, multiplies them by corresponding weights, sums these weighted inputs, adds a bias, and then passes the result through an activation function to produce an output.

#### Key Components:

1.  **Inputs (X)**: These are the features or data points fed into the perceptron.

2.  **Weights (W)**: Each input `xᵢ` is associated with a weight `wᵢ`. Weights determine the strength or importance of each input. During training, the perceptron learns optimal weight values.

3.  **Bias (b)**: The bias is an additional parameter that allows the perceptron to shift the activation function curve up or down. It helps the model fit data better by allowing it to activate even when all inputs are zero.

4.  **Weighted Sum (z)**: This is the sum of the products of inputs and their corresponding weights, plus the bias.

$$z = (x_1 \cdot w_1) + (x_2 \cdot w_2) + ... + (x_n \cdot w_n) + b$$

    In vector form, this can be written as:
$$\mathbf{z} = \mathbf{X} \cdot \mathbf{W}^T + b$$

5.  **Activation Function**: This function introduces non-linearity into the perceptron. It transforms the weighted sum into the final output. For binary classification, it typically squashes the output into a specific range (e.g., 0 to 1). We will use the **Sigmoid** activation function for this analysis.

    The Sigmoid function is defined as:
$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

    It maps any real-valued number into a range between 0 and 1, making it suitable for probabilities or binary classification where values close to 0 or 1 indicate the class.

#### How it works:

The perceptron calculates the weighted sum of its inputs and applies the activation function. If the output exceeds a certain threshold (or based on the activation function's output), it classifies the input into one category; otherwise, it classifies it into another.

In [1]:
import numpy as np

In [2]:
def sigmoid(x):
    """Sigmoid activation function."""
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    """Derivative of the sigmoid function."""
    return x * (1 - x)

### Perceptron Class Structure

Now, let's define our `Perceptron` class. This class will encapsulate the logic for initializing weights and biases, performing the forward pass (prediction), and handling the backpropagation (learning) process.

**Initialization (`__init__`)**:
- We'll initialize weights and bias with small random values. This helps break symmetry and allows the model to learn different features.
- The number of input features determines the size of the weight vector.

**Forward Pass (`forward`)**:
- This method will take input features, calculate the weighted sum, add the bias, and then apply the sigmoid activation function to produce the output prediction.

**Training (`train`)**:
- This method will iteratively adjust the weights and bias using gradient descent. For each training iteration (epoch), it will perform the forward pass to get predictions, calculate the error, and then use the error to update weights and bias based on the learning rate.

In [3]:
class Perceptron:
    def __init__(self, num_inputs, learning_rate=0.1):
        # Initialize weights randomly with a small scale
        # Adding 1 to num_inputs for the bias weight if we treat bias as a weight with a constant input of 1
        # Or, we can initialize bias separately. Let's initialize bias separately for clarity.
        self.weights = np.random.uniform(size=(num_inputs, 1)) * 0.01 # Small random weights
        self.bias = np.random.uniform(size=(1, 1)) * 0.01 # Small random bias
        self.learning_rate = learning_rate

        print(f"Perceptron initialized with {num_inputs} inputs:")
        print(f"  Initial Weights shape: {self.weights.shape}")
        print(f"  Initial Bias shape: {self.bias.shape}")

    def forward(self, inputs):
        # Calculate the weighted sum of inputs plus bias
        # inputs: (num_samples, num_inputs)
        # weights: (num_inputs, 1)
        # dot product result: (num_samples, 1)
        self.z = np.dot(inputs, self.weights) + self.bias
        # Apply the sigmoid activation function
        self.activation = sigmoid(self.z)
        return self.activation

    def train(self, training_inputs, training_outputs, epochs):
        print(f"\nStarting training for {epochs} epochs...")
        for epoch in range(epochs):
            # Forward pass
            predictions = self.forward(training_inputs)

            # Calculate the error (difference between actual and predicted)
            error = training_outputs - predictions

            # Calculate the delta for weights and bias using the sigmoid derivative
            # This is where backpropagation comes into play for a single layer
            d_predictions = error * sigmoid_derivative(predictions)

            # Update weights
            # Transpose training_inputs to (num_inputs, num_samples) for matrix multiplication
            # d_predictions is (num_samples, 1)
            # Resulting weight_update is (num_inputs, 1)
            weight_update = np.dot(training_inputs.T, d_predictions)
            self.weights += self.learning_rate * weight_update

            # Update bias
            # Bias update is the sum of d_predictions for all samples
            bias_update = np.sum(d_predictions, axis=0)
            self.bias += self.learning_rate * bias_update

            if (epoch + 1) % 10 == 0 or epoch == 0:
                loss = np.mean(np.square(error)) # Mean Squared Error as a loss metric
                print(f"Epoch {epoch + 1}/{epochs}, Loss: {loss:.4f}")

        print("Training complete!")
        print(f"  Final Weights shape: {self.weights.shape}")
        print(f"  Final Bias shape: {self.bias.shape}")
        print(f"  Final Weights:\n{self.weights}")
        print(f"  Final Bias:\n{self.bias}")

### Training the Perceptron: The Learning Process

The `train` method is where the perceptron learns from data. This process involves repeatedly showing the perceptron examples, calculating its error, and adjusting its weights and bias to reduce that error. This is a form of **gradient descent**.

#### Key Steps in Training:

1.  **Epochs**: Training is done over a number of `epochs`. An epoch represents one full pass through the entire training dataset.

2.  **Forward Pass**: For each input in the training data, the perceptron first performs a `forward` pass to predict an output. This prediction is generated using the current weights and bias.

3.  **Error Calculation**: The `error` is calculated as the difference between the `training_output` (the true label) and the `predictions` made by the perceptron. A common metric is the Mean Squared Error (MSE), but for updating parameters, we primarily use the raw difference.

    $$Error = Y_{true} - Y_{predicted}$$

4.  **Backpropagation (Weight and Bias Update)**:
    This is the core of the learning process. We use the error to determine how much to adjust the `weights` and `bias`. This adjustment is based on the **gradient** of the error with respect to the weights and bias.

    -   **Delta for Predictions (`d_predictions`)**: We calculate this by multiplying the `error` by the `sigmoid_derivative` of the `predictions`. The derivative tells us how much the output changes with respect to its input, which helps in scaling the error appropriately.

        $$\delta_{output} = Error \times \sigma'(z)$$
        where $\sigma'(z) = \sigma(z) \cdot (1 - \sigma(z))$ is the derivative of the sigmoid function, which we implemented as `sigmoid_derivative(predictions)`.

    -   **Weight Update**: The weights are updated proportionally to the `d_predictions` and the original `training_inputs`. This means that inputs that contributed more to a wrong prediction will have their corresponding weights adjusted more significantly.

        $$W_{new} = W_{old} + \text{learning_rate} \times (\text{training_inputs}^T \cdot \delta_{output})$$

    -   **Bias Update**: The bias is updated based on the sum of `d_predictions` across all training samples. It adjusts the overall threshold for activation.

        $$b_{new} = b_{old} + \text{learning_rate} \times \sum(\delta_{output})$$

5.  **Learning Rate**: The `learning_rate` controls how large the steps are during weight and bias adjustments. A high learning rate can lead to instability, while a very low one can make training slow.

This iterative process allows the perceptron to gradually learn the underlying patterns in the data, minimizing the error and improving its classification accuracy over time.

### Data Preparation for Training

To train our perceptron, we need a dataset with input features (X) and corresponding output labels (Y). For this demonstration, we'll create a simple binary classification problem.

We will define a dataset where:
-   `X` represents two input features (e.g., `x1`, `x2`).
-   `Y` represents the target output, which will be either 0 or 1, indicating the class.

Let's create a dataset that represents a simple AND-like or OR-like gate, which is a common example for demonstrating perceptrons.

In [4]:
# Define the training data
# Each row is a sample, each column is an input feature
# For simplicity, let's create a dataset for an OR gate logic
X = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
])

# Corresponding output labels (0 or 1)
Y = np.array([
    [0],
    [1],
    [1],
    [1]
])

print("Input Features (X):")
print(X)
print("\nTarget Outputs (Y):")
print(Y)

# Instantiate the Perceptron
# num_inputs should match the number of features in X
num_input_features = X.shape[1]
perceptron = Perceptron(num_input_features, learning_rate=0.1)

Input Features (X):
[[0 0]
 [0 1]
 [1 0]
 [1 1]]

Target Outputs (Y):
[[0]
 [1]
 [1]
 [1]]
Perceptron initialized with 2 inputs:
  Initial Weights shape: (2, 1)
  Initial Bias shape: (1, 1)


### Training the Perceptron

Now that we have our data and the perceptron instantiated, we can proceed with training. We'll specify a number of `epochs` (iterations) for the perceptron to learn from the data. During each epoch, the perceptron will adjust its weights and bias to minimize the prediction error.

In [5]:
# Train the perceptron
# We'll use a significant number of epochs to allow it to converge
epochs = 1000
perceptron.train(X, Y, epochs)


Starting training for 1000 epochs...
Epoch 1/1000, Loss: 0.2476
Epoch 10/1000, Loss: 0.1883
Epoch 20/1000, Loss: 0.1589
Epoch 30/1000, Loss: 0.1442
Epoch 40/1000, Loss: 0.1352
Epoch 50/1000, Loss: 0.1287
Epoch 60/1000, Loss: 0.1234
Epoch 70/1000, Loss: 0.1187
Epoch 80/1000, Loss: 0.1145
Epoch 90/1000, Loss: 0.1105
Epoch 100/1000, Loss: 0.1068
Epoch 110/1000, Loss: 0.1032
Epoch 120/1000, Loss: 0.0998
Epoch 130/1000, Loss: 0.0966
Epoch 140/1000, Loss: 0.0935
Epoch 150/1000, Loss: 0.0905
Epoch 160/1000, Loss: 0.0877
Epoch 170/1000, Loss: 0.0850
Epoch 180/1000, Loss: 0.0824
Epoch 190/1000, Loss: 0.0799
Epoch 200/1000, Loss: 0.0776
Epoch 210/1000, Loss: 0.0753
Epoch 220/1000, Loss: 0.0731
Epoch 230/1000, Loss: 0.0711
Epoch 240/1000, Loss: 0.0691
Epoch 250/1000, Loss: 0.0672
Epoch 260/1000, Loss: 0.0654
Epoch 270/1000, Loss: 0.0637
Epoch 280/1000, Loss: 0.0620
Epoch 290/1000, Loss: 0.0604
Epoch 300/1000, Loss: 0.0589
Epoch 310/1000, Loss: 0.0574
Epoch 320/1000, Loss: 0.0560
Epoch 330/1000, 

### Making Predictions and Evaluating Performance

After training, the perceptron should have learned appropriate weights and bias to correctly classify the training data. We can now use its `forward` method to make predictions on new (or the same) data and evaluate how well it performs.

Since it's a binary classifier, we'll typically apply a threshold (e.g., 0.5) to the sigmoid output to get discrete class labels (0 or 1).

In [6]:
# Make predictions after training
predictions = perceptron.forward(X)

# Apply a threshold to get binary output (0 or 1)
binary_predictions = (predictions > 0.5).astype(int)

print("\n--- Evaluation ---")
print("Input Features (X):\n", X)
print("\nActual Outputs (Y):\n", Y)
print("\nPredicted Outputs (binary):\n", binary_predictions)

# Calculate accuracy
accuracy = np.mean(binary_predictions == Y) * 100
print(f"\nAccuracy: {accuracy:.2f}%")

print("\nFinal Learned Weights:\n", perceptron.weights)
print("Final Learned Bias:\n", perceptron.bias)

# Let's try predicting for a new, unseen input (e.g., [0,0])
# This is already in our training data, but demonstrates the prediction step
new_input = np.array([[0, 0]])
new_prediction_raw = perceptron.forward(new_input)
new_prediction_binary = (new_prediction_raw > 0.5).astype(int)
print(f"\nPrediction for input {new_input[0]}: Raw = {new_prediction_raw[0][0]:.4f}, Binary = {new_prediction_binary[0][0]}")

new_input = np.array([[1, 0]])
new_prediction_raw = perceptron.forward(new_input)
new_prediction_binary = (new_prediction_raw > 0.5).astype(int)
print(f"Prediction for input {new_input[0]}: Raw = {new_prediction_raw[0][0]:.4f}, Binary = {new_prediction_binary[0][0]}")

new_input = np.array([[0.5, 0.5]]) # A point not exactly in training data
new_prediction_raw = perceptron.forward(new_input)
new_prediction_binary = (new_prediction_raw > 0.5).astype(int)
print(f"Prediction for input {new_input[0]}: Raw = {new_prediction_raw[0][0]:.4f}, Binary = {new_prediction_binary[0][0]}")


--- Evaluation ---
Input Features (X):
 [[0 0]
 [0 1]
 [1 0]
 [1 1]]

Actual Outputs (Y):
 [[0]
 [1]
 [1]
 [1]]

Predicted Outputs (binary):
 [[0]
 [1]
 [1]
 [1]]

Accuracy: 100.00%

Final Learned Weights:
 [[3.3083405 ]
 [3.30835671]]
Final Learned Bias:
 [[-1.34506504]]

Prediction for input [0 0]: Raw = 0.2067, Binary = 0
Prediction for input [1 0]: Raw = 0.8769, Binary = 1
Prediction for input [0.5 0.5]: Raw = 0.8769, Binary = 1


### Summary and Conclusion

In this analysis, we successfully built a single-layer perceptron for binary classification from scratch. We covered:

1.  **Theoretical Foundation**: Understanding the components of a perceptron (inputs, weights, bias, activation function).
2.  **Implementation**: Defining the sigmoid activation function and a `Perceptron` class with `__init__`, `forward`, and `train` methods.
3.  **Training Process**: Explaining how the perceptron learns through iterative updates of weights and bias using gradient descent and the derivative of the sigmoid function.
4.  **Demonstration**: Creating a simple dataset (OR gate logic), training the perceptron on it, and evaluating its performance, showing that it can learn to correctly classify the given patterns.

This basic perceptron forms the foundation for more complex neural network architectures. It demonstrates the core principles of how a neural network learns to map inputs to outputs by adjusting its internal parameters.